# Extras (opcionales) — Proyecto Final LanusStats · v2

**Lucas Marinelli · @datafutbol_ar**

| § | Extra | Output |
|---|---|---|
| 1 | Módulo reusable `pases_progresivos.py` | importado y demo |
| 2 | Replicar **PyPizza** de Messi | `extras_pizza_messi.png` |

### Fix v2 (sobre v1)
- El módulo `pases_progresivos.py` se escribió **directamente al disco** (no desde el notebook) para evitar errores de escape de comillas que rompían la función.
- Acá lo **importamos** y verificamos que funciona.


## Setup

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
from helpers import *
import matplotlib.pyplot as plt
from mplsoccer import PyPizza
from tqdm.auto import tqdm

# Importamos la función del módulo independiente
from pases_progresivos import agregar_pases_progresivos

partidos = lista_partidos()
print('Función importada del módulo:', agregar_pases_progresivos.__name__)
print('Docstring (primera línea):', agregar_pases_progresivos.__doc__.split(chr(10))[0])


---

# § Extra 1 — Pases progresivos como módulo reusable

## Qué quedó en disco

El archivo `pases_progresivos.py` está **al lado de este notebook**, así que
cualquier otro notebook del proyecto puede usarlo con:

```python
from pases_progresivos import agregar_pases_progresivos
```

## Definición técnica

Un **pase progresivo** = pase **completado** + avanza **≥10 m** hacia el arco rival.

En StatsBomb el campo va de `x=0` (arco propio) a `x=120` (arco rival),
así que la condición es:

```python
es_progresivo = (pass_outcome is NaN) AND (x_end - x >= 10)
```


## Demo de uso — Argentina vs Arabia Saudita

In [ ]:
ev = cargar_eventos(MATCH_ARG_SAU, 'arg_sau')

# Filtrar pases de Argentina
pases_arg = ev[(ev['type'] == 'Pass') & (ev['team'] == 'Argentina')]
print(f'Pases ARG en el partido: {len(pases_arg)}')

# Aplicar la función importada
pases_arg_con_prog = agregar_pases_progresivos(pases_arg)

# Verificación
total = len(pases_arg_con_prog)
completados = pases_arg_con_prog['es_completado'].sum()
progresivos = pases_arg_con_prog['es_progresivo'].sum()

print(f'\nARG vs SAU — Argentina:')
print(f'  Pases totales:       {total}')
print(f'  Pases completados:   {completados} ({completados/total*100:.1f}%)')
print(f'  Pases progresivos:   {progresivos} ({progresivos/completados*100:.1f}% de los completados)')
print()
print('Top 5 jugadores por pases progresivos en este partido:')
top = (pases_arg_con_prog[pases_arg_con_prog['es_progresivo']]
       .groupby('player').size().sort_values(ascending=False).head(5))
print(top.to_string())


## Verificación rápida

Si la celda anterior te imprime algo como:
```
Pases progresivos: 130 (24.0% de los completados)
```
significa que la función está funcionando. (En el v1 fallido daba 0 porque las comillas escaparon mal y rompieron el filtro.)


---

# § Extra 2 — PyPizza de Messi (Mundial 2022)

Replica el [tutorial oficial de mplsoccer PyPizza](https://mplsoccer.readthedocs.io/en/latest/gallery/pizza_plots/index.html)
usando datos reales del torneo y la paleta Combo C de @datafutbol_ar.


## Cargar el torneo completo

In [ ]:
ev_mundial = []
for mid in tqdm(partidos['match_id'].tolist(), desc='Cargando'):
    try:
        e = cargar_eventos(mid, f'mundial_{mid}')
        e['match_id'] = mid
        ev_mundial.append(e)
    except Exception as ex:
        print(f'⚠️ {mid}: {ex}')

ev_total = pd.concat(ev_mundial, ignore_index=True)
ev_total = añadir_xy(ev_total)
ev_reg = ev_total[ev_total['period'] <= 4].copy()
print(f'{len(ev_reg):,} eventos · {ev_reg["match_id"].nunique()} partidos')


## Calcular las 10 métricas por jugador

In [ ]:
def metricas_completas(ev_reg):
    tiros = ev_reg[ev_reg['type'] == 'Shot']
    pases_all = ev_reg[ev_reg['type'] == 'Pass']
    # Aprovechamos la función del módulo
    pases_completos = agregar_pases_progresivos(pases_all)

    df = pd.DataFrame({
        'goles':         tiros[tiros['shot_outcome'] == 'Goal'].groupby('player').size(),
        'xg':            tiros.groupby('player')['shot_statsbomb_xg'].sum().round(2),
        'tiros':         tiros.groupby('player').size(),
        'pases_comp':    pases_completos[pases_completos['es_completado']].groupby('player').size(),
        'pases_prog':    pases_completos[pases_completos['es_progresivo']].groupby('player').size(),
        'regates_ok':    ev_reg[(ev_reg['type'] == 'Dribble') &
                                 (ev_reg['dribble_outcome'] == 'Complete')].groupby('player').size(),
        'recup':         ev_reg[ev_reg['type'] == 'Ball Recovery'].groupby('player').size(),
        'faltas_recib':  ev_reg[ev_reg['type'] == 'Foul Won'].groupby('player').size(),
        'partidos':      ev_reg.groupby('player')['match_id'].nunique(),
    }).fillna(0)

    pases_int = pases_all.groupby('player').size()
    pases_ok = pases_completos[pases_completos['es_completado']].groupby('player').size()
    df['acierto_pct'] = ((pases_ok / pases_int) * 100).fillna(0).round(1)
    return df


df_jug = metricas_completas(ev_reg)
print(f'Total jugadores: {len(df_jug)}')

# Grupo de comparación: ofensivos (≥10 tiros en el torneo)
comp = df_jug[df_jug['tiros'] >= 10].copy()
comp['xg_por_tiro'] = (comp['xg'] / comp['tiros']).round(3)
print(f'Grupo de comparación (≥10 tiros): {len(comp)}')


## Calcular percentiles de Messi vs el grupo

In [ ]:
from scipy.stats import percentileofscore

# Las 10 métricas para el PyPizza
METRICAS_PIZZA = [
    ('goles',         'Goles'),
    ('xg',            'xG total'),
    ('tiros',         'Tiros'),
    ('xg_por_tiro',   'xG por tiro'),    # propia
    ('pases_comp',    'Pases completados'),
    ('acierto_pct',   '% acierto pases'),
    ('pases_prog',    'Pases progresivos'),
    ('regates_ok',    'Regates exitosos'),
    ('faltas_recib',  'Faltas recibidas'),
    ('recup',         'Recuperaciones'),
]

JUGADOR = 'Lionel Andrés Messi Cuccittini'

percentiles_messi = []
valores_messi = []
labels = []

for col, label in METRICAS_PIZZA:
    valor = comp.loc[JUGADOR, col] if JUGADOR in comp.index else 0
    pct = percentileofscore(comp[col], valor, kind='rank')
    percentiles_messi.append(round(pct))
    valores_messi.append(valor)
    labels.append(label)

# Verificación con barra ASCII
print(f'PERCENTILES de Messi vs {len(comp)} jugadores ofensivos del Mundial 2022:')
print()
for l, v, p in zip(labels, valores_messi, percentiles_messi):
    barra = '█' * (p // 5)
    print(f'  {l:22s} {v:>8.2f}  →  P{p:>3.0f} {barra}')


## Generar el PyPizza

In [ ]:
def pizza_messi(labels, percentiles, archivo):
    # 4 ataque + 3 distribución + 3 otras
    slice_colors = ([COLORS['accent']] * 4 +
                    [COLORS['primary']] * 3 +
                    ['#009E73'] * 3)
    text_colors = [COLORS['text']] * 10

    baker = PyPizza(
        params=labels,
        background_color=COLORS['bg'],
        straight_line_color=COLORS['muted'],
        straight_line_lw=1,
        last_circle_color=COLORS['text'],
        last_circle_lw=1.5,
        other_circle_lw=0,
        inner_circle_size=15,
    )

    fig, ax = baker.make_pizza(
        percentiles,
        figsize=(10, 11),
        color_blank_space='same',
        slice_colors=slice_colors,
        value_colors=text_colors,
        value_bck_colors=slice_colors,
        blank_alpha=0.4,
        kwargs_slices=dict(edgecolor=COLORS['text'], zorder=2, linewidth=1.2),
        kwargs_params=dict(color=COLORS['text'], fontsize=11,
                           fontweight='bold', va='center'),
        kwargs_values=dict(
            color=COLORS['text'], fontsize=10, fontweight='bold', zorder=3,
            bbox=dict(edgecolor=COLORS['text'], facecolor=COLORS['bg'],
                      boxstyle='round,pad=0.2', lw=1)
        ),
    )
    fig.patch.set_facecolor(COLORS['bg'])

    fig.text(0.515, 0.97, 'LIONEL MESSI · MUNDIAL 2022',
             size=22, ha='center', color=COLORS['text'], weight='bold')
    fig.text(0.515, 0.935,
             f'Percentiles vs jugadores ofensivos del Mundial 2022 (≥10 tiros · n={len(comp)})',
             size=11, ha='center', color=COLORS['accent'], style='italic')

    fig.text(0.12, 0.025, '● Ataque',         color=COLORS['accent'],  size=10, weight='bold')
    fig.text(0.40, 0.025, '● Distribución',   color=COLORS['primary'], size=10, weight='bold')
    fig.text(0.72, 0.025, '● Otras',          color='#009E73',          size=10, weight='bold')

    fig.text(0.02, 0.01, '@datafutbol_ar', color=COLORS['accent'],
             size=9, weight='bold')
    fig.text(0.98, 0.01, 'Datos: StatsBomb Open Data · Plot: mplsoccer PyPizza',
             color=COLORS['muted'], size=8, style='italic', ha='right')

    guardar_fig(fig, archivo)
    plt.show()


pizza_messi(labels, percentiles_messi, 'extras_pizza_messi')


---

## Resumen — Extras ✅ v2

| § | Entregable | Archivo |
|---|---|---|
| 1 | Módulo reusable Python | `pases_progresivos.py` |
| 1 | Demo aplicada al partido del debut | output en celdas |
| 2 | PyPizza Messi Mundial 2022 | `extras_pizza_messi.png` |

### Diferencias respecto a v1

| | v1 (fallido) | v2 (este) |
|---|---|---|
| Cómo se creó el módulo .py | Desde celda del notebook con string heredoc → comillas escapadas mal | Escrito **directo al disco** con código Python limpio |
| Resultado del demo | 0 pases progresivos (función rota) | ~130 pases progresivos (la cifra esperada) |
| `Path` import | Faltaba en la celda | Importado al inicio del setup |

### Insight del pizza

El pizza de Messi va a estar **dominado por el dorado** (porciones de ataque grandes:
goles, xG, tiros, xG/tiro) y también con celeste grande en pases progresivos. La porción
más chica probablemente sea **recuperaciones** (no es su rol) — eso confirma que el
gráfico está bien calibrado (un Messi con 100% en defensa sería sospechoso).

---


